# Solar filament segmentation: reproduce training and inference

This notebook uses the versioned source in `src/solarseg`, pinned dependencies in `requirements.txt`, and official competition files. Read `docs/competition_contract.md` first. The notebook has not been executed as part of authoring; result cells show measured files only when you run the corresponding stage.

Accept the Kaggle rules and authenticate yourself. Download with `scripts/download_data.sh`. Do not download public full-MAGFiLO annotations: some observations overlap the competition test set. No test labels or external ground-truth metadata enter this pipeline.

Launch this notebook with a kernel whose working directory is `notebooks/`. If your notebook service starts its kernel at the repository root, explicitly set `PROJECT_ROOT = Path.cwd()` in the next cell. Moving the project folder does not require machine-specific source paths.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

PROJECT_ROOT = Path("..").resolve()
PYTHON_BIN = sys.executable
DEVICE = "mps"  # Explicit choices: "mps", "cuda", "cpu".
RUN_NAME = "notebook_reproduced"
RUN_TRAINING = False  # Set True to prepare, train CV, finalize and create predictions.
RUN_INFERENCE = False  # Set True for inference from the saved model selected below.

print({"project": str(PROJECT_ROOT), "python": PYTHON_BIN, "device": DEVICE})


## Install and acquire data

In the selected Python environment, install `requirements.txt` before running the cells below. The CLI and notebook share the same source and data contract:

```text
data/raw/MAGFiLO_1.0_Kaggle_2026/
  train/train_images/*.jpeg
  train/MAGFiLO_1.0_Annotations_kaggle2026_train.json
  test/test_images/*.jpeg
```

The data download script does not accept rules or log in for you. It downloads the official archive after you perform those actions.

## Full reproducible training

The script writes a new `runs/<RUN_NAME>/neural.json` with the selected device. It keeps the frozen configuration untouched and fails if the run directory already exists. Use a fresh run name for another complete experiment.

Preparation constructs one row per source image, independently encodes each annotator's instances for PQ, and creates soft-consensus training masks. Four contiguous year blocks keep fold3 locked. Development training and validation use a 27-day embargo around the holdout, and training also excludes observations near its current validation block. The exact IDs and one-time audit are saved in the processed directory.

The six fixed classical candidates and neural postprocessing/50% fusion candidates are selected on development PQ. Finalization freezes that choice before evaluating holdout once; it then fits the selected neural configuration on all official training observations. No holdout-driven re-selection occurs.


In [ ]:
if RUN_TRAINING:
    training_env = dict(os.environ)
    training_env.update(PYTHON_BIN=PYTHON_BIN, DEVICE=DEVICE, RUN_NAME=RUN_NAME)
    subprocess.run(
        ["bash", "scripts/reproduce.sh"],
        cwd=PROJECT_ROOT,
        env=training_env,
        check=True,
    )


## Inspect saved experiment evidence

These cells read the results produced by the training run; they do not rerun scoring. Scores are local measurements, not Kaggle leaderboard scores. The run folders also retain per-image measurements, source hashes, logs, configurations and model checkpoints.


In [ ]:
if RUN_TRAINING:
    run_root = PROJECT_ROOT / "runs" / RUN_NAME
    for relative in (
        "classical_v1/scores.json",
        "unet_v1/scores.json",
        "final_v1/selected_before_holdout.json",
        "final_v1/holdout_score.json",
    ):
        print(relative)
        print(json.dumps(json.loads((run_root / relative).read_text()), indent=2))


## Inference from trained weights, without retraining

Keep `RUN_TRAINING=False`, set `RUN_INFERENCE=True`, and point `MODEL_FILE` at an existing `model.json`. For the project's first completed run use `models/final_v1/model.json`; for this notebook's training use `runs/notebook_reproduced/final_v1/model.json`.

Keep the referenced checkpoint at the relative path recorded in that JSON. A classical selection needs only its parameters. The CLI reads official test H-alpha images, restores full 2048×2048 masks, performs instance postprocessing and writes compressed COCO ASCII RLE. It validates the generated file once and records its model/hash/status beside the CSV. Images with no detections receive no fabricated empty instance.

Choose a new `SUBMISSION_FILE` name: existing submissions are not overwritten. No command in this notebook uploads a submission to Kaggle.


In [ ]:
MODEL_FILE = Path("models/final_v1/model.json")
SUBMISSION_FILE = Path("submissions/notebook_inference_v1.csv")

if RUN_INFERENCE:
    inference_env = dict(os.environ, PYTHONPATH=str(PROJECT_ROOT / "src"))
    (PROJECT_ROOT / SUBMISSION_FILE.parent).mkdir(parents=True, exist_ok=True)
    subprocess.run(
        [
            PYTHON_BIN, "-u", "-m", "solarseg.cli", "infer",
            "--model", str(MODEL_FILE),
            "--output", str(SUBMISSION_FILE),
            "--device", DEVICE,
        ],
        cwd=PROJECT_ROOT,
        env=inference_env,
        check=True,
    )


In [ ]:
if RUN_INFERENCE:
    record = json.loads((PROJECT_ROOT / SUBMISSION_FILE.with_suffix(".json")).read_text())
    print(json.dumps({key: value for key, value in record.items() if key != "images"}, indent=2))


## Interpretation and reproducibility

The official PQ counts pairs with strictly `IoU > 0.5` separately for each annotator-image record, then pools `sum(IoU)/(TP+0.5FP+0.5FN)`. It is not the mean of per-image PQ and does not replace official pairing with Hungarian matching. Training consensus masks are not substituted for the independent-annotator evaluation truth.

The submission sidecar starts with `kaggle_status=not_uploaded` and null public/private scores. Local contract success does not establish Kaggle acceptance or a leaderboard score. Upload and any resulting official score must be recorded explicitly.

Fixed seeds, dependencies and source hashes allow the experiment to be reconstructed. CPU, MPS and CUDA floating-point differences can change training trajectories; cross-device bitwise identity is not promised. The small model and finite candidate list are an initial comparison, and the small time-out holdout does not establish broad observatory or solar-cycle generalization.
